Tarea 04

Física Computacional

Edwin Urbina Quiroz|A76588

## Parte I: Memoria Compartida

### a. Estrategia de paralelización

El script C++ `fourier.cpp` tiene un bucle `for`. Para optimizar el cómputo, se ejecutará dicho bucle dentro de una región paralela. Además, se dividirá el bucle entre los diferentes hilos disponibles.

Con respecto a los accesos de memoria concurridos y posibles condiciones de carrera:

- reduction(+: sum): OpenMP no realiza reducción sobre complejos en C++. Además, como `sum` está declarado dentro del bucle `for(k)`, no es una variable común, eso quiere que no lo accesan todos los hilos al mismo tiempo, sino que cada hilo tiene su propia suma y no hay condición de carrera.

- Existe la variable común `num_procs`, por lo que podría ser reescrita por varios hilos simultáneamente. Para evitar condición de carrera, se puede proteger y hacer que solo la accece el hilo master con `#pragma omp master`.

- Podría también haber escritura concurrente en output[$k$] si varios hilos escribieran el mismo índice $k$. No es el caso porque el `#pragma op for` reparte las iteraciones sin que haya traslape.

### b. Implementación de paralelización

Se ha agregado una región de paralelización con `#pragma omp parallel` y un `#pragma omp for` para dividir el bucle `for` entre los distintos hilos.

### c. Evaluación de la aceleración

El script bash `scalingFourier.sh` compila el script C++ `fourier.cpp`, ejecuta el `fourier.x` con OMP_NUM_THREADS = {1, ..., MAX_THREADS} y redirige las salidas (número de hilos, tiempo) hacia `scaling.dat`.

Luego, con base en el archivo `scaling.dat`, calcula los valores de la aceleración $S_P$ al hacer $T_1 / T_P$ y redirecciona los resulados hacia `speedupFourier.dat`.

### d.Gráfico de la aceleración vs hilos

El script bash `scalingFourier.sh` ejecuta el script GNUplot `fig.plt`, que a su vez se encarga de plotear los datos almacenados en `speedupFourier.dat` y genera la salida `Figure.pdf`.

## Parte II: Memoria Distribuida

Se siguieron las indicaciones dadas en el enunciado de la tarea. A continuación explicamos el uso de `MPI_Scatter` y de `MPI_Reduce`. Por lo demás, el código en `dot_product.cpp` está propiamente documentado.

### a.MPI_Scatter

`MPI_Scatter` distribuye los datos a cada proceso. Recibe 8 parámetros como sigue:
 - 3 del send (datos a distribuir, número de elementos a enviar a cada proceso, tipo de los datos)
 - 3 del recv (receptor de los datos, número de elementos que recibirá cada proceso, tipo de los datos)
 - 1 del proceso raíz
 - 1 el communicador

En nuestro caso, los parámetros son:

Del send:
- vector a distribuir (a)
- número de elementos a enviar a cada proceso (nlocal)
- tipo de los datos (MPI_DOUBLE)

Del recv:
- receptor de los datos (local_a.data())
- número de elementos que recibirá cada proceso (nlocal)
- tipo de los datos (MPI_DOUBLE)

Proceso raíz:
- 0

Comunicador:
- MPI_COMM_WORLD

Para el vector a:

```C++
MPI_Scatter( a.data(), 
            nlocal,
            MPI_DOUBLE,
            local_a.data(),
            nlocal,
            MPI_DOUBLE,
            0,
            MPI_COMM_WORLD);
            
```

Similar para el vector b.

Para más detalles, véase [la documentación en rookiehpc](https://rookiehpc.org/mpi/docs/mpi_scatter/index.html).

### b.MPI_Reduce

`MPI_Reduce` colecta el cómputo efectuado por cada proceso y los combita. Recibe 7 parámetros como sigue:
 - 1 del send (el resultado local de cada proceso)
 - 1 del recv (el acumulador)
 - número de elementos a reducir
 - tipo de datos
 - la operación que queremos reducir
 - el proceso raíz que almacena la reducción
 - el comunicador

En nuestro caso, los parámetros son:
- local_result
- total_result
- 1
- MPI_DOUBLE
- MPI_SUM
- 0
- MPI_COMM_WORLD

```C++
MPI_Reduce(
    &local_result,
    &total_result,
    1,
    MPI_DOUBLE,
    MPI_SUM,
    0,
    MPI_COMM_WORLD);
    
```

Para más detalles, véase [la documentación en rookiehpc](https://rookiehpc.org/mpi/docs/mpi_reduce/index.html).